# Chapter 13: Large Language Models — Putting It All Together

An LLM is just:

```
Tokenizer (Ch 10) → Embedding (Ch 11) → Transformer × N (Ch 12) → Predict next token
```

That's it. The "magic" comes from **scale**:
- Billions of parameters
- Trillions of tokens of training data
- Thousands of GPUs for weeks

This chapter shows how the pieces connect and how an LLM generates text.

---
## The Full Architecture

```
"The cat sat" → what comes next?

Step 1: Tokenize        → ["The", "cat", "sat"]     → [464, 3797, 3332]
Step 2: Embed + Position → [[0.2, -0.1, ...], [0.8, 0.3, ...], [0.5, 0.7, ...]]
Step 3: Transformer ×N  → each layer refines understanding
Step 4: Output head      → probability for EVERY token in vocabulary
Step 5: Sample           → pick one token → "down"
Step 6: Repeat           → "The cat sat down" → predict next...
```

In [ ]:
import numpy as np

np.random.seed(42)

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Tiny LLM: vocab=10, d_model=8, 1 layer
vocab = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4,
         "dog": 5, "ran": 6, "down": 7, ".": 8, "[END]": 9}
idx_to_word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)
d_model = 8

# Model components (all random — untrained)
embedding_table = np.random.randn(vocab_size, d_model) * 0.3
W_Q = np.random.randn(d_model, d_model) * 0.1
W_K = np.random.randn(d_model, d_model) * 0.1
W_V = np.random.randn(d_model, d_model) * 0.1
W_out = np.random.randn(d_model, vocab_size) * 0.1  # output head

def tiny_llm_forward(token_ids):
    """Forward pass of a tiny LLM."""
    # Step 2: Embed
    X = embedding_table[token_ids]
    
    # Step 3: Self-attention (1 layer, 1 head, with causal mask)
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores = Q @ K.T / np.sqrt(d_model)
    # Causal mask
    mask = np.triu(np.ones((len(token_ids), len(token_ids))), k=1) * -1e9
    scores += mask
    weights = softmax(scores)
    X = weights @ V
    
    # Step 4: Output head → logits for each position
    logits = X @ W_out  # shape: (seq_len, vocab_size)
    
    # We only care about the LAST position (predicting next token)
    last_logits = logits[-1]
    probs = softmax(last_logits)
    
    return probs

# Run it
input_text = "the cat sat"
input_ids = [vocab[w] for w in input_text.split()]

print("Tiny LLM Forward Pass:\n")
print(f"  Input:     '{input_text}'")
print(f"  Token IDs: {input_ids}")

probs = tiny_llm_forward(input_ids)

print(f"\n  Predicted next token probabilities:")
sorted_idx = np.argsort(-probs)
for idx in sorted_idx:
    bar = "█" * int(probs[idx] * 50)
    print(f"    {idx_to_word[idx]:<6} {probs[idx]:>6.1%}  {bar}")

print(f"\n  (Random weights → random predictions. Training fixes this.)")

---
## Training: Next Token Prediction

The training objective is stunningly simple:

**Given all previous tokens, predict the next token.**

```
"The"              → predict "cat"      (learn that animals follow "the")
"The cat"          → predict "sat"      (learn that verbs follow nouns)
"The cat sat"      → predict "on"       (learn common phrases)
"The cat sat on"   → predict "the"      (learn grammar patterns)
```

Do this on trillions of tokens from the internet, and the model learns grammar, facts, reasoning, and more — all from this one simple task.

In [ ]:
# Demonstrate training on a tiny corpus
corpus = [
    [0, 1, 2, 3, 0, 4, 8],  # the cat sat on the mat .
    [0, 5, 6, 7, 8],         # the dog ran down .
    [0, 1, 6, 8],            # the cat ran .
    [0, 5, 2, 3, 0, 4, 8],  # the dog sat on the mat .
]

print("Training: Next Token Prediction\n")
print("  From the sentence 'the cat sat on the mat .':\n")

sentence = corpus[0]
sentence_words = [idx_to_word[i] for i in sentence]

print(f"  {'Input (context)':<30} {'Target (next token)':>20}")
print(f"  {'─'*30} {'─'*20}")
for i in range(1, len(sentence)):
    context = " ".join(sentence_words[:i])
    target = sentence_words[i]
    print(f"  {context:<30} {target:>20}")

print(f"\n  One sentence produces {len(sentence)-1} training examples!")
print(f"  A book with 100K tokens → 100K training examples.")
print(f"  The entire internet → trillions of examples.")

In [ ]:
# Actually train our tiny LLM
np.random.seed(42)

# Reinitialize
emb = np.random.randn(vocab_size, d_model) * 0.3
Wq = np.random.randn(d_model, d_model) * 0.1
Wk = np.random.randn(d_model, d_model) * 0.1
Wv = np.random.randn(d_model, d_model) * 0.1
Wo = np.random.randn(d_model, vocab_size) * 0.1
lr = 0.01

def forward_and_loss(token_ids):
    """Forward pass returning loss and intermediate values."""
    seq_len = len(token_ids) - 1  # last token is target only
    X = emb[token_ids[:seq_len]]
    
    Q = X @ Wq; K = X @ Wk; V = X @ Wv
    scores = Q @ K.T / np.sqrt(d_model)
    mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -1e9
    attn_w = softmax(scores + mask)
    attn_out = attn_w @ V
    
    logits = attn_out @ Wo
    
    # Loss: cross-entropy at each position
    total_loss = 0
    for t in range(seq_len):
        probs = softmax(logits[t])
        target = token_ids[t + 1]
        total_loss -= np.log(probs[target] + 1e-8)
    
    return total_loss / seq_len, logits

print("Training our tiny LLM...\n")
print(f"  {'Epoch':>6} {'Loss':>8} {'Sample prediction'}")
print(f"  {'─'*6} {'─'*8} {'─'*30}")

for epoch in range(501):
    total_loss = 0
    
    for sentence in corpus:
        loss, logits = forward_and_loss(sentence)
        total_loss += loss
        
        # Simple gradient descent via numerical gradients
        # (Real training uses backprop, but this shows the concept)
        eps = 0.001
        for param in [Wo]:  # only update output weights for speed
            grad = np.zeros_like(param)
            for i in range(min(param.shape[0], d_model)):
                for j in range(min(param.shape[1], vocab_size)):
                    param[i, j] += eps
                    loss_plus, _ = forward_and_loss(sentence)
                    param[i, j] -= 2 * eps
                    loss_minus, _ = forward_and_loss(sentence)
                    param[i, j] += eps
                    grad[i, j] = (loss_plus - loss_minus) / (2 * eps)
            param -= lr * grad
    
    if epoch % 100 == 0:
        # Test: predict after "the cat"
        test_ids = [0, 1]  # the cat
        X = emb[test_ids]
        Q = X @ Wq; K = X @ Wk; V = X @ Wv
        scores = Q @ K.T / np.sqrt(d_model)
        mask_t = np.triu(np.ones((2, 2)), k=1) * -1e9
        attn_out = softmax(scores + mask_t) @ V
        probs = softmax(attn_out[-1] @ Wo)
        pred = idx_to_word[np.argmax(probs)]
        avg_loss = total_loss / len(corpus)
        print(f"  {epoch:>6} {avg_loss:>8.3f} 'the cat' → '{pred}' ({probs[np.argmax(probs)]:.1%})")

print(f"\n  (This tiny model can only learn simple patterns.")
print(f"   Real LLMs train on billions of examples with backprop.)")

---
## Text Generation: How LLMs Write

Generation is just repeated next-token prediction:

```
Input:  "The cat"
Step 1: "The cat" → predict "sat"     → "The cat sat"
Step 2: "The cat sat" → predict "on"  → "The cat sat on"
Step 3: "The cat sat on" → predict "the" → "The cat sat on the"
...
Until [END] token or max length
```

This is called **autoregressive** generation.

In [ ]:
# Autoregressive generation (simulated with pre-set probabilities)
# Since our tiny model isn't well-trained, we'll simulate a trained one

# Simulated probability distributions (what a trained model would output)
trained_probs = {
    (0,):       {1: 0.45, 5: 0.45, 4: 0.10},            # the → cat/dog/mat
    (0, 1):     {2: 0.50, 6: 0.40, 7: 0.10},            # the cat → sat/ran/down
    (0, 5):     {6: 0.50, 2: 0.40, 7: 0.10},            # the dog → ran/sat/down
    (0, 1, 2):  {3: 0.60, 7: 0.30, 8: 0.10},            # the cat sat → on/down/.
    (0, 1, 6):  {7: 0.50, 3: 0.30, 8: 0.20},            # the cat ran → down/on/.
    (0, 5, 6):  {7: 0.50, 3: 0.30, 8: 0.20},            # the dog ran → down/on/.
    (0, 1, 2, 3): {0: 0.90, 8: 0.10},                   # the cat sat on → the
    (0, 1, 2, 7): {8: 0.90, 3: 0.10},                   # the cat sat down → .
}

def generate(start_tokens, max_len=8):
    """Generate text autoregressively."""
    tokens = list(start_tokens)
    steps = []
    
    for _ in range(max_len):
        key = tuple(tokens)
        # Try progressively shorter context
        probs = None
        for start in range(len(key)):
            subkey = key[start:]
            if subkey in trained_probs:
                probs = trained_probs[subkey]
                break
        
        if probs is None:
            probs = {8: 1.0}  # default: end
        
        # Sample from distribution
        token_ids = list(probs.keys())
        token_probs = list(probs.values())
        next_token = np.random.choice(token_ids, p=token_probs)
        
        context = " ".join(idx_to_word[t] for t in tokens)
        pred_word = idx_to_word[next_token]
        prob = probs[next_token]
        steps.append((context, pred_word, prob, probs))
        
        tokens.append(next_token)
        if next_token == 9 or next_token == 8:  # [END] or .
            break
    
    return tokens, steps

# Generate several times to show randomness
print("Autoregressive Generation:\n")

np.random.seed(42)
tokens, steps = generate([0, 1])  # start with "the cat"

print(f"Starting with: 'the cat'\n")
print(f"  {'Step':>4} {'Context':<25} {'Choices':<35} {'Picked'}")
print(f"  {'─'*4} {'─'*25} {'─'*35} {'─'*10}")

for i, (context, pred, prob, all_probs) in enumerate(steps):
    choices = ", ".join(f"{idx_to_word[k]}({v:.0%})" for k, v in sorted(all_probs.items(), key=lambda x: -x[1]))
    print(f"  {i+1:>4} {context:<25} {choices:<35} {pred} ({prob:.0%})")

result = " ".join(idx_to_word[t] for t in tokens)
print(f"\n  Result: '{result}'")

# Generate multiple times
print(f"\nMultiple generations (same start, different randomness):")
for trial in range(5):
    tokens, _ = generate([0, 1])
    result = " ".join(idx_to_word[t] for t in tokens)
    print(f"  {trial+1}. '{result}'")

print("\n  Same input, different outputs — because we SAMPLE from probabilities.")

---
## Temperature: Controlling Randomness

**Temperature** controls how random the generation is:

```
Temperature = 0.0: always pick the most likely token (deterministic)
Temperature = 1.0: sample from the original probabilities
Temperature = 2.0: more random, creative, sometimes nonsensical
```

Technically: `softmax(logits / temperature)`

Low temperature → sharpens probabilities (confident)
High temperature → flattens probabilities (random)

In [ ]:
# Temperature effect
logits = np.array([2.0, 1.0, 0.5, 0.1, -0.5])
token_names = ["sat", "ran", "on", "down", "."]

print("Temperature Effect on Token Probabilities:\n")
print(f"  Raw logits: {logits}\n")

for temp in [0.1, 0.5, 1.0, 1.5, 3.0]:
    probs = softmax(logits / temp)
    print(f"  Temperature = {temp}:")
    for name, prob in zip(token_names, probs):
        bar = "█" * int(prob * 40)
        print(f"    {name:<6} {prob:>6.1%}  {bar}")
    print()

print("  T=0.1: almost always picks 'sat' (highest logit)")
print("  T=3.0: nearly uniform — could pick anything")
print("\n  ChatGPT/Claude typically use T≈0.7 for conversation.")

---
## Top-k and Top-p Sampling

Temperature alone can still produce nonsense. Additional filters:

- **Top-k**: only consider the k most likely tokens
- **Top-p (nucleus)**: only consider tokens whose cumulative probability reaches p

In [ ]:
# Top-k and Top-p
probs = softmax(logits)
sorted_idx = np.argsort(-probs)
sorted_probs = probs[sorted_idx]
sorted_names = [token_names[i] for i in sorted_idx]

print("Sampling Strategies:\n")
print("  Full distribution:")
cumsum = 0
for name, prob in zip(sorted_names, sorted_probs):
    cumsum += prob
    bar = "█" * int(prob * 40)
    print(f"    {name:<6} {prob:>6.1%}  (cumulative: {cumsum:.1%})  {bar}")

print("\n  Top-k = 3 (only top 3 tokens):")
for i, (name, prob) in enumerate(zip(sorted_names, sorted_probs)):
    if i < 3:
        print(f"    {name:<6} ✓ included")
    else:
        print(f"    {name:<6} ✗ filtered out")

print("\n  Top-p = 0.9 (tokens until 90% cumulative):")
cumsum = 0
for name, prob in zip(sorted_names, sorted_probs):
    cumsum += prob
    if cumsum <= 0.92:  # slightly over to include the boundary token
        print(f"    {name:<6} ✓ included  (cumulative: {cumsum:.1%})")
    else:
        print(f"    {name:<6} ✗ filtered out")

print("\n  In practice: Top-p = 0.9 or 0.95 is the most common setting.")

---
## Pre-training vs Fine-tuning vs RLHF

Building ChatGPT/Claude is a 3-stage process:

```
Stage 1: PRE-TRAINING (this chapter)
  Data:   Trillions of tokens from the internet
  Task:   Predict next token
  Result: "Base model" — knows language, facts, code
          But just completes text, doesn't follow instructions

Stage 2: FINE-TUNING (Ch 15)
  Data:   Thousands of (instruction, response) pairs
  Task:   Learn to follow instructions
  Result: Can answer questions, but may be unhelpful/unsafe

Stage 3: RLHF (Ch 14)
  Data:   Human rankings of model responses
  Task:   Learn which responses humans prefer
  Result: Helpful, harmless, honest assistant (ChatGPT/Claude)
```

In [ ]:
print("The Three Stages of Building an AI Assistant:\n")

stages = [
    ("Stage 1: Pre-training",
     "Predict next token on internet text",
     '  User: "What is the capital of France?"\n'
     '  Model: "What is the capital of Germany? What is the capital of Italy?"\n'
     '  (It just continues the pattern — not answering!)'),
    ("Stage 2: Fine-tuning",
     "Train on instruction→response pairs",
     '  User: "What is the capital of France?"\n'
     '  Model: "The capital of France is Paris."\n'
     '  (Answers correctly, but might also say harmful things)'),
    ("Stage 3: RLHF",
     "Learn from human preference rankings",
     '  User: "What is the capital of France?"\n'
     '  Model: "The capital of France is Paris. It\'s located in the\n'
     '          north-central part of the country along the Seine River."\n'
     '  (Helpful, informative, safe)'),
]

for name, method, example in stages:
    print(f"  {name}")
    print(f"  Method: {method}")
    print(f"  Example:")
    print(f"{example}")
    print()

---
## The Scaling Laws

A key discovery: LLM performance scales predictably with:
1. **Parameters** (model size)
2. **Data** (training tokens)
3. **Compute** (GPU hours)

More of any → better performance. This is why the AI race is about scale.

In [ ]:
print("LLM Scaling Over Time:\n")
print(f"  {'Year':>5} {'Model':<18} {'Parameters':>12} {'Training Tokens':>16}")
print(f"  {'─'*5} {'─'*18} {'─'*12} {'─'*16}")

models = [
    (2018, "GPT-1",          "117M",      "~5B"),
    (2019, "GPT-2",          "1.5B",      "~40B"),
    (2020, "GPT-3",          "175B",      "300B"),
    (2022, "PaLM",           "540B",      "780B"),
    (2023, "LLaMA-2",        "70B",       "2T"),
    (2023, "GPT-4",          "~1.8T?",    "~13T?"),
    (2024, "LLaMA-3",        "405B",      "15T"),
]

for year, name, params, tokens in models:
    print(f"  {year:>5} {name:<18} {params:>12} {tokens:>16}")

print("\n  Parameters grew ~15,000× in 6 years (117M → 1.8T).")
print("  Training data grew ~3,000× (5B → 15T tokens).")
print("\n  But architecture? Barely changed. Still transformers.")
print("  The insight: scale the SAME architecture with MORE data.")

---
## What Does the Model Actually "Know"?

An LLM is a **compressed representation** of its training data.

It doesn't store facts in a database. Instead, facts are encoded in the **weights** — the same weights that also encode grammar, reasoning patterns, and style.

```
"Paris" is connected to "France" through weight patterns,
not through a lookup table that says capital_of[France] = Paris.
```

This is why LLMs can:
- Combine facts in new ways (reasoning)
- Generate plausible-sounding but wrong facts (hallucination)
- Know things implicitly without being able to cite sources

In [ ]:
print("What LLMs Can and Cannot Do:\n")

print("  CAN:")
capabilities = [
    "Generate fluent text in many languages",
    "Answer questions about trained knowledge",
    "Write and explain code",
    "Reason about simple logic problems",
    "Translate between languages",
    "Summarize long documents",
    "Follow complex multi-step instructions",
]
for c in capabilities:
    print(f"    ✓ {c}")

print("\n  CANNOT:")
limitations = [
    "Access the internet in real-time (frozen at training cutoff)",
    "Truly understand or have consciousness",
    "Reliably do precise math (it's pattern matching, not calculating)",
    "Guarantee factual accuracy (hallucination)",
    "Learn from a conversation permanently (no memory between sessions)",
]
for l in limitations:
    print(f"    ✗ {l}")

print("\n  The key insight: LLMs are NEXT TOKEN PREDICTORS.")
print("  Everything they do emerges from that one simple ability,")
print("  applied at massive scale.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Architecture** | Tokenize → Embed → Transformer × N → Predict next token |
| **Training** | Predict next token on trillions of tokens |
| **Generation** | Repeated next-token prediction (autoregressive) |
| **Temperature** | Controls randomness (low=deterministic, high=creative) |
| **Top-k / Top-p** | Filter unlikely tokens before sampling |
| **3 Stages** | Pre-training → Fine-tuning → RLHF |
| **Scaling** | Same architecture, more params + data = better |
| **Hallucination** | Plausible but wrong — fundamental limitation |

**Next chapter**: RLHF — how a raw text predictor becomes a helpful assistant